# Final Project: Supermart Grocery Sales – Story & Insights

This notebook presents the **key insights** from the Supermart Grocery Sales dataset through a focused narrative and interactive Plotly visualizations. The story centers on understanding **what drives profitability** — across categories, regions, time, and discount strategies — to prepare actionable insights for a dashboard.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

COLOR_SALES = '#2196F3'
COLOR_PROFIT = '#4CAF50'
COLOR_DISCOUNT = '#FF9800'
COLOR_NEGATIVE = '#F44336'
TEMPLATE = 'plotly_white'

In [2]:
df = pd.read_csv('Supermart Grocery Sales - Retail Analytics Dataset.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', dayfirst=False)
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Month_Name'] = df['Order Date'].dt.month_name()
df['Year_Month'] = df['Order Date'].dt.to_period('M').dt.to_timestamp()
df['Profit_Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

print(f'Dataset: {df.shape[0]:,} orders | {df["Order Date"].min().strftime("%b %Y")} – {df["Order Date"].max().strftime("%b %Y")}')

Dataset: 9,994 orders | Jan 2015 – Dec 2018


---
## Story Overview

**Central question:** *What drives profitability at Supermart, and where are the biggest opportunities?*

We'll answer this through four lenses:
1. **Business Growth** — How have sales and profit evolved over time?
2. **Category Profitability** — Which product categories deliver the best returns?
3. **Regional Performance** — Which regions are the profit engines?
4. **Discount Strategy** — Are discounts helping or hurting the bottom line?

---
## 1. Business Growth Over Time

First, let's look at the big picture — how has Supermart's revenue and profitability trended?

In [3]:
monthly = df.groupby('Year_Month').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'count')
).reset_index()

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Scatter(x=monthly['Year_Month'], y=monthly['Sales'], name='Sales',
               line=dict(color=COLOR_SALES, width=2), fill='tozeroy', fillcolor='rgba(33,150,243,0.1)'),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=monthly['Year_Month'], y=monthly['Profit'], name='Profit',
               line=dict(color=COLOR_PROFIT, width=2)),
    secondary_y=True
)

fig.update_layout(
    title='Monthly Sales & Profit Trend',
    template=TEMPLATE, height=450, hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.update_yaxes(title_text='Sales', secondary_y=False)
fig.update_yaxes(title_text='Profit', secondary_y=True)
fig.show()

In [4]:
yearly = df.groupby('Year').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'count'),
    Avg_Discount=('Discount', 'mean')
).reset_index()
yearly['Profit_Margin'] = (yearly['Profit'] / yearly['Sales'] * 100).round(2)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Yearly Sales & Profit', 'Yearly Profit Margin'))

fig.add_trace(go.Bar(x=yearly['Year'], y=yearly['Sales'], name='Sales', marker_color=COLOR_SALES), row=1, col=1)
fig.add_trace(go.Bar(x=yearly['Year'], y=yearly['Profit'], name='Profit', marker_color=COLOR_PROFIT), row=1, col=1)
fig.add_trace(go.Scatter(x=yearly['Year'], y=yearly['Profit_Margin'], name='Margin %',
                         mode='lines+markers', marker=dict(size=10), line=dict(color=COLOR_NEGATIVE, width=2.5)),
              row=1, col=2)

fig.update_layout(template=TEMPLATE, height=400, showlegend=True,
                  legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='right', x=1))
fig.update_yaxes(title_text='Amount', row=1, col=1)
fig.update_yaxes(title_text='Profit Margin (%)', row=1, col=2)
fig.show()

> **Insight:** Sales show a general upward trend year over year, with notable monthly fluctuations. Profit largely tracks sales, indicating consistent margins — but certain periods show divergence worth investigating.

---
## 2. Category Profitability — Where the Money Comes From

Not all product categories are equally profitable. Let's compare revenue generation against actual margin performance.

In [5]:
cat = df.groupby('Category').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'count'),
    Avg_Discount=('Discount', 'mean')
).reset_index()
cat['Profit_Margin'] = (cat['Profit'] / cat['Sales'] * 100).round(2)
cat = cat.sort_values('Sales', ascending=True)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Revenue vs Profit by Category', 'Profit Margin by Category'),
                    shared_yaxes=True)

fig.add_trace(go.Bar(y=cat['Category'], x=cat['Sales'], name='Sales',
                     orientation='h', marker_color=COLOR_SALES), row=1, col=1)
fig.add_trace(go.Bar(y=cat['Category'], x=cat['Profit'], name='Profit',
                     orientation='h', marker_color=COLOR_PROFIT), row=1, col=1)

colors = [COLOR_PROFIT if m > cat['Profit_Margin'].median() else COLOR_DISCOUNT for m in cat['Profit_Margin']]
fig.add_trace(go.Bar(y=cat['Category'], x=cat['Profit_Margin'], name='Margin %',
                     orientation='h', marker_color=colors, showlegend=False), row=1, col=2)

fig.update_layout(template=TEMPLATE, height=450, barmode='group',
                  legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='right', x=0.45))
fig.update_xaxes(title_text='Amount', row=1, col=1)
fig.update_xaxes(title_text='Profit Margin (%)', row=1, col=2)
fig.show()

In [6]:
sub_cat = df.groupby(['Category', 'Sub Category']).agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum')
).reset_index()
sub_cat['Profit_Margin'] = (sub_cat['Profit'] / sub_cat['Sales'] * 100).round(2)

fig = px.treemap(sub_cat, path=['Category', 'Sub Category'], values='Sales',
                 color='Profit_Margin', color_continuous_scale='RdYlGn',
                 color_continuous_midpoint=sub_cat['Profit_Margin'].median(),
                 title='Category & Sub-Category Treemap (size = Sales, color = Profit Margin %)')
fig.update_layout(template=TEMPLATE, height=500)
fig.show()

> **Insight:** High-revenue categories don't necessarily have the best margins. The treemap reveals sub-categories where sales volume is large but profitability lags — these are prime targets for pricing or discount strategy adjustments.

---
## 3. Regional Performance — Geography of Profit

How do the four regions compare? Which ones generate the most value, and are there efficiency differences?

In [7]:
region = df.groupby('Region').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'count'),
    Avg_Discount=('Discount', 'mean')
).reset_index()
region['Profit_Margin'] = (region['Profit'] / region['Sales'] * 100).round(2)
region['Avg_Order_Value'] = (region['Sales'] / region['Orders']).round(2)

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=('Total Sales by Region', 'Profit Margin by Region', 'Avg Order Value by Region'))

fig.add_trace(go.Bar(x=region['Region'], y=region['Sales'], marker_color=COLOR_SALES, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=region['Region'], y=region['Profit_Margin'],
                     marker_color=[COLOR_PROFIT if m > region['Profit_Margin'].median() else COLOR_DISCOUNT
                                   for m in region['Profit_Margin']], showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=region['Region'], y=region['Avg_Order_Value'],
                     marker_color='#9C27B0', showlegend=False), row=1, col=3)

fig.update_layout(template=TEMPLATE, height=400)
fig.update_yaxes(title_text='Sales', row=1, col=1)
fig.update_yaxes(title_text='Margin (%)', row=1, col=2)
fig.update_yaxes(title_text='Avg Order Value', row=1, col=3)
fig.show()

In [8]:
region_cat = df.groupby(['Region', 'Category']).agg(Sales=('Sales', 'sum')).reset_index()

fig = px.bar(region_cat, x='Region', y='Sales', color='Category',
             title='Sales Composition by Region & Category',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(template=TEMPLATE, height=450, barmode='stack',
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

> **Insight:** Regions differ not just in total volume but also in margin efficiency and category mix. Some regions carry higher average discounts that compress margins despite strong sales — a clear opportunity for targeted discount policies.

---
## 4. Discount Strategy — The Profitability Trade-off

Discounts drive volume, but at what cost? Let's quantify the relationship between discount levels and profitability.

In [9]:
fig = px.scatter(df, x='Discount', y='Profit_Margin', color='Category',
                 opacity=0.4, size_max=8,
                 title='Discount vs Profit Margin (by Category)',
                 color_discrete_sequence=px.colors.qualitative.Set2)
fig.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='Break-even')
fig.update_layout(template=TEMPLATE, height=450)
fig.update_xaxes(title_text='Discount Rate')
fig.update_yaxes(title_text='Profit Margin (%)')
fig.show()

In [10]:
df['Discount_Bin'] = pd.cut(df['Discount'], bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5],
                            labels=['0-10%', '10-20%', '20-30%', '30-40%', '40-50%'])

disc = df.groupby('Discount_Bin', observed=True).agg(
    Avg_Profit=('Profit', 'mean'),
    Avg_Margin=('Profit_Margin', 'mean'),
    Total_Profit=('Profit', 'sum'),
    Orders=('Order ID', 'count')
).reset_index()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Avg Profit by Discount Level', 'Avg Profit Margin by Discount Level'))

bar_colors = [COLOR_PROFIT if v > 0 else COLOR_NEGATIVE for v in disc['Avg_Profit']]
fig.add_trace(go.Bar(x=disc['Discount_Bin'].astype(str), y=disc['Avg_Profit'],
                     marker_color=bar_colors, showlegend=False), row=1, col=1)

margin_colors = [COLOR_PROFIT if v > disc['Avg_Margin'].median() else COLOR_DISCOUNT for v in disc['Avg_Margin']]
fig.add_trace(go.Bar(x=disc['Discount_Bin'].astype(str), y=disc['Avg_Margin'],
                     marker_color=margin_colors, showlegend=False), row=1, col=2)

fig.update_layout(template=TEMPLATE, height=400)
fig.update_yaxes(title_text='Avg Profit', row=1, col=1)
fig.update_yaxes(title_text='Avg Margin (%)', row=1, col=2)
fig.show()

In [11]:
disc_region = df.groupby(['Region', 'Discount_Bin'], observed=True).agg(
    Avg_Margin=('Profit_Margin', 'mean')
).reset_index()

fig = px.bar(disc_region, x='Discount_Bin', y='Avg_Margin', color='Region',
             barmode='group', title='Discount Impact on Margin by Region',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.update_layout(template=TEMPLATE, height=420,
                  xaxis_title='Discount Level', yaxis_title='Avg Profit Margin (%)')
fig.show()

> **Insight:** There's a clear tipping point — moderate discounts (10-20%) maintain healthy margins, while aggressive discounts (30%+) substantially erode profitability. The pattern holds across regions, though some regions tolerate discounts better than others.

---
## 5. Seasonality & Monthly Patterns

Understanding when peak sales occur helps with inventory and marketing planning.

In [12]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

monthly_avg = df.groupby('Month_Name').agg(
    Avg_Sales=('Sales', 'mean'),
    Avg_Profit=('Profit', 'mean'),
    Orders=('Order ID', 'count')
).reindex(month_order)

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=monthly_avg['Avg_Sales'].tolist() + [monthly_avg['Avg_Sales'].iloc[0]],
    theta=month_order + [month_order[0]],
    fill='toself', name='Avg Sales',
    line=dict(color=COLOR_SALES)
))
fig.add_trace(go.Scatterpolar(
    r=monthly_avg['Avg_Profit'].tolist() + [monthly_avg['Avg_Profit'].iloc[0]],
    theta=month_order + [month_order[0]],
    fill='toself', name='Avg Profit',
    line=dict(color=COLOR_PROFIT)
))
fig.update_layout(template=TEMPLATE, height=500, title='Seasonal Pattern: Avg Sales & Profit by Month',
                  polar=dict(radialaxis=dict(visible=True)),
                  legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5))
fig.show()

In [13]:
heatmap_data = df.groupby(['Year', 'Month']).agg(Sales=('Sales', 'sum')).reset_index()
heatmap_pivot = heatmap_data.pivot(index='Year', columns='Month', values='Sales')
heatmap_pivot.columns = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig = px.imshow(heatmap_pivot, text_auto='.0f', aspect='auto',
                color_continuous_scale='Blues',
                title='Sales Heatmap: Year × Month')
fig.update_layout(template=TEMPLATE, height=350, xaxis_title='Month', yaxis_title='Year')
fig.show()

> **Insight:** Seasonal peaks and troughs are visible, with certain months consistently outperforming others. This pattern can inform inventory stocking and promotional timing.

---
## 6. Top Performers — Customers & Cities

Identifying the highest-value customers and locations helps focus retention and expansion strategies.

In [14]:
top_cust = df.groupby('Customer Name').agg(
    Sales=('Sales', 'sum'), Profit=('Profit', 'sum'), Orders=('Order ID', 'count')
).nlargest(10, 'Sales').reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(y=top_cust['Customer Name'], x=top_cust['Sales'],
                     name='Sales', orientation='h', marker_color=COLOR_SALES))
fig.add_trace(go.Bar(y=top_cust['Customer Name'], x=top_cust['Profit'],
                     name='Profit', orientation='h', marker_color=COLOR_PROFIT))

fig.update_layout(template=TEMPLATE, height=420, barmode='group',
                  title='Top 10 Customers by Sales',
                  xaxis_title='Amount', yaxis=dict(autorange='reversed'),
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

In [15]:
top_city = df.groupby('City').agg(
    Sales=('Sales', 'sum'), Profit=('Profit', 'sum'), Orders=('Order ID', 'count')
).reset_index()
top_city['Profit_Margin'] = (top_city['Profit'] / top_city['Sales'] * 100).round(2)
top_city = top_city.nlargest(12, 'Sales')

fig = px.scatter(top_city, x='Sales', y='Profit_Margin', size='Orders',
                 text='City', color='Profit_Margin',
                 color_continuous_scale='RdYlGn',
                 title='Top 12 Cities: Sales vs Profit Margin (bubble size = order count)')
fig.update_traces(textposition='top center')
fig.update_layout(template=TEMPLATE, height=480,
                  xaxis_title='Total Sales', yaxis_title='Profit Margin (%)')
fig.show()

> **Insight:** A few high-value customers and cities contribute disproportionately to overall revenue. Some top-revenue cities have lower-than-expected margins, suggesting localized discount overuse or cost structure differences.

---
## Summary & Dashboard Direction

### Key Takeaways

| Dimension | Finding |
|---|---|
| **Growth** | Sales and profit are growing year-over-year, with seasonal fluctuations |
| **Categories** | High-revenue categories can have surprisingly thin margins; sub-category analysis reveals optimization targets |
| **Regions** | Not all regions are equal — some combine high volume with strong margins, while others over-discount |
| **Discounts** | Discounts above ~20-30% sharply erode profitability; the sweet spot is 10-20% |
| **Customers** | Top customers deserve retention strategies; city-level performance varies widely |

### Dashboard Plan

The final dashboard should include:
- **KPI cards** for total sales, profit, margin, and order count
- **Time-series view** with monthly trends and year-over-year comparison
- **Category breakdown** with treemap and margin comparison
- **Regional performance** panel with filters
- **Discount impact** analysis to guide pricing decisions
- **Top performers** table for customers and cities